# Winning Jeopardy

Jeopardy is a popular TV show in the US where participants answer questions to win money.
I am analyzing the Jeopardy dataset to figure out the patterns in the questions to maximize the winning probability

In [1]:
import pandas as pd

In [2]:
# Reading the contents of the file into a Dataframe
jeopardy=pd.read_csv("jeopardy.csv")

In [3]:
#Getting the first 5 rows of the "jeopardy" DataFrame
jeopardy.head(5)

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams


In [4]:
# Deeply understanding the format of the columns in jeopardy dataframe

jeopardy.columns

Index(['Show Number', ' Air Date', ' Round', ' Category', ' Value',
       ' Question', ' Answer'],
      dtype='object')

Looks like, there are spaces in the column names in the dataset. So, removing the additional spaces in the column names, for easy retrieval.

In [5]:
# Removing the spaces at the prefix and suffix of the column names
jeopardy.columns=jeopardy.columns.str.strip()

In [6]:
jeopardy.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer'],
      dtype='object')

Now the extra spaces at the from and end of the column names are trimmed out.

Do we know whether the same question is repeated? 
Do we know if the questions differ only by some additional punctuations marks or spaces?
Do we know if the case difference is making the questions different?

Writing a function to standardize the question and the answer column values.

In [7]:
import re
def normalize_text(text):
    text=re.sub(r"[^a-zA-Z0-9\s]","",text.lower().strip())
    return(text)


In [8]:
jeopardy["clean_question"]=jeopardy["Question"].apply(normalize_text)

In [9]:
jeopardy["clean_answer"]=jeopardy["Answer"].apply(normalize_text)

In [10]:
jeopardy.head(5)

,Show Number,Air Date,Round,Category,Value,Question,Answer,clean_question,clean_answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus,for the last 8 years of his life galileo was u...,copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe,no 2 1912 olympian football star at carlisle i...,jim thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona,the city of yuma in this state has a record av...,arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's,in 1963 live on the art linkletter show this c...,mcdonalds
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams,signer of the dec of indep framer of the const...,john adams


In [11]:
jeopardy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19999 entries, 0 to 19998
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Show Number     19999 non-null  int64 
 1   Air Date        19999 non-null  object
 2   Round           19999 non-null  object
 3   Category        19999 non-null  object
 4   Value           19663 non-null  object
 5   Question        19999 non-null  object
 6   Answer          19999 non-null  object
 7   clean_question  19999 non-null  object
 8   clean_answer    19999 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.4+ MB


Price Value of the Question and Air Date columns are objects meaning, though they have integer and date values respectively, it is not defined according to the values. Hence , fixing the values according to the datatype. Also, "Value" column has some null values.

In [12]:
import numpy as np
def value_fix(cost):
    cost=re.sub(r"[$,]","",str(cost))
    if cost=="" or cost=="nan":
        return 0
    else:
        return(int(cost))
    

In [13]:
jeopardy["clean_value"]=jeopardy["Value"].apply(value_fix)

In [14]:
jeopardy["Air Date"]=pd.to_datetime(jeopardy["Air Date"])

In [15]:
jeopardy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19999 entries, 0 to 19998
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Show Number     19999 non-null  int64         
 1   Air Date        19999 non-null  datetime64[ns]
 2   Round           19999 non-null  object        
 3   Category        19999 non-null  object        
 4   Value           19663 non-null  object        
 5   Question        19999 non-null  object        
 6   Answer          19999 non-null  object        
 7   clean_question  19999 non-null  object        
 8   clean_answer    19999 non-null  object        
 9   clean_value     19999 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(7)
memory usage: 1.5+ MB


We could see that, Air Date column is now a Datetime column and Value column is cleaned to clean_value column of integer datatype.

In [16]:
splitted_text="in 1963 live on the art linkletter show this".split()
splitted_text

['in', '1963', 'live', 'on', 'the', 'art', 'linkletter', 'show', 'this']

In [17]:
def split_row(row):
    split_answer=row["clean_answer"].split()
    split_question=row["clean_question"].split()
    match_count=0
    split_answer=[item for item in split_answer if item != "the"]
    if len(split_answer)==0:
        return 0
    else:
        for item in split_answer:
            if item in split_question:
                match_count += 1
    return(match_count/len(split_answer))
    

In [18]:
answer_in_question = jeopardy.apply(split_row, axis=1)

In [19]:
answer_in_question.mean()

np.float64(0.05834744478926688)

Only 5% of the answers are used in the Questions itself,It means Studying Past Answers will not work, and the answers are based on reasoning.

In [20]:
question_overlap=[]
terms_used=set()

In [21]:
jeopardy.sort_values(by="Air Date",inplace=True)

In [22]:

for index,row in jeopardy.iterrows():
    match_count = 0
    split_question=row["clean_question"].split()
    split_question=[item for item in split_question if  len(item)>6]
    for item in split_question:
        if item in terms_used:
            match_count += 1
        else:
            terms_used.add(item)
    if len(split_question)>0:
        question_overlap.append(match_count/len(split_question))
    else:
        question_overlap.append(0)

print(sum(question_overlap)/len(question_overlap))    
        
        
        
        
        

0.623382467987927


78 % of the values are repeated in the question. So, It means, We can study the old questions to improve the winning chance.

### Defining Hypothesis

Null Hypothesis H0 Occurence of repeated words in the questions is independent of high and low value questions
Alternate Hypothesis H1 - Occurence of repeated words in the questions is dependent of high and low value questions

In [23]:
# Labelling High Value and Low Value questions

def label_value(row):
    if row["clean_value"]>800:
        return 1
    else:
        return 0

jeopardy["high_value"]=jeopardy.apply(label_value,axis=1)
jeopardy["high_value"].value_counts()

high_value
0    14265
1     5734
Name: count, dtype: int64

In [24]:
# Finding to know if the word is present in how many low value questions and how many high-value questions
def value_counts(word):
    low_count=0
    high_count=0
    for index,row in jeopardy.iterrows():
        if word in row["clean_question"].split():
            if row["high_value"]==1:
                high_count +=1
            else:
                low_count +=1
    return(high_count,low_count)
                
                
                
            

In [25]:
# Instead of running the above function over the entire list of words, picking randomly 10 items without duplicates
import random
comparison_terms=random.sample(list(terms_used),10)

In [26]:
comparison_terms

['jagiello',
 'singapore',
 'chasing',
 'homespun',
 'dollars',
 'different',
 'comparative',
 'collective',
 'exclusive',
 'onetime']

In [27]:
observed_expected=[]
for item in comparison_terms:
    observed_expected.append(value_counts(item))

In [28]:
observed_expected

[(0, 1),
 (2, 1),
 (0, 2),
 (0, 1),
 (1, 5),
 (9, 28),
 (0, 2),
 (1, 7),
 (2, 0),
 (2, 3)]

In [29]:
high_value_count=len(jeopardy[jeopardy["high_value"]==1])
low_value_count=len(jeopardy[jeopardy["high_value"]==0])

In [30]:
high_value_count,low_value_count

(5734, 14265)

In [36]:
import numpy as np
from scipy.stats import chisquare

chi_squared = []
total_rows = len(jeopardy)

for high, low in observed_expected:
    total = high + low
    
    if total == 0:   # skip words that never appear
        chi_squared.append((np.nan, np.nan))
        continue

    total_prop = total / total_rows
    expected_high_value_rows = total_prop * high_value_count
    expected_low_value_rows  = total_prop * low_value_count

    # Chi-square: f_obs should be [high, low], not [high_value_count, low_value_count]
    chisq, p = chisquare(f_obs=[high, low],
                         f_exp=[expected_high_value_rows, expected_low_value_rows])
    
    chi_squared.append((chisq, p))

In [37]:
chi_squared

[(np.float64(0.401962846126884), np.float64(0.5260772985705469)),
 (np.float64(2.1177104383031944), np.float64(0.14560406868263753)),
 (np.float64(0.803925692253768), np.float64(0.3699222378079571)),
 (np.float64(0.401962846126884), np.float64(0.5260772985705469)),
 (np.float64(0.42281054506129573), np.float64(0.515537958129453)),
 (np.float64(0.34189277990072214), np.float64(0.5587387057840683)),
 (np.float64(0.803925692253768), np.float64(0.3699222378079571)),
 (np.float64(1.0229964471766237), np.float64(0.31180929640924016)),
 (np.float64(4.97558423439135), np.float64(0.025707519787911092)),
 (np.float64(0.3137668167849311), np.float64(0.5753778622944691))]

Looking at the computed chi-square statistics and associated p-values for the sampled words, the p-values are mostly greater than 0.05. This indicates that the observed differences between the expected and observed counts are negligible. Therefore, we cannot reject the null hypothesis, and we conclude that the occurrence of words in questions is largely independent of whether the question is high-value or low-value.